# Task 6 — Idempotent replay verification

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | `screenshots/replay/stage3_replay_manifest.json` |
| Command | `bash scripts/finalize_stage3_evidence.sh` |
| Replay target | `src/datasets/__init__.py` |
| Publication behavior | The cells consume captured artifacts and never connect to live services during a Pages build. |
```

## Approach and reasoning

Command: `bash scripts/finalize_stage3_evidence.sh`. The canonical Stage 3 workflow rebuilds the full-repository baseline, restarts Spark with the existing checkpoint, modifies only `src/datasets/__init__.py`, replays that file with a new `run_id`, waits for Kafka Connect and Spark, removes stale graph entities, and validates the final stores. The notebook reads the strict replay manifest so the public book stays deterministic.

## What this chapter proves

| Requirement | Evidence in this chapter |
|---|---|
| Idempotent replay | The executed cells compare baseline, replay, and final-state artifacts. |
| Checkpoint reuse | Kafka/Spark checkpoint output is shown from the captured run. |
| Store convergence | Neo4j cleanup and MongoDB replacement are validated with JSON artifacts and UI screenshots. |


In [1]:
from pathlib import Path
import json

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'screenshots/replay/stage3_replay_manifest.json').exists())
MANIFEST_PATH = ROOT / 'screenshots/replay/stage3_replay_manifest.json'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['stage'] == 3 and manifest['status'] == 'pass'
print('manifest:', MANIFEST_PATH.relative_to(ROOT))
print('captured_at:', manifest['run']['captured_at'])


manifest: screenshots\replay\stage3_replay_manifest.json
captured_at: 2026-07-23T06:56:48Z


## Modified file and run identity

This section pins the replay to one baseline file and one replay `run_id`, so the later store checks can distinguish expected append-log repetition from duplicate database state.


In [2]:
source = manifest['source']
print('target:', source['target_file'])
print('file_id:', source['file_id'])
print('content_hash:', source['before_content_hash'], '->', source['after_content_hash'])
print('run_id:', manifest['run']['baseline_run_id'], '->', manifest['run']['replay_run_id'])
assert source['before_content_hash'] != source['after_content_hash']
assert manifest['run']['baseline_run_id'] != manifest['run']['replay_run_id']
assert source['source_restored'] is True


target: src/datasets/__init__.py
file_id: 6c39568a6a11c430
content_hash: 74ab176247c9ba61e09843280c18cf3a9ac690403a5d35411a75d097db721939 -> 6db67191532027fbf91190815efa70380d79c2eda9e61136a41a1d1a8a8e0735
run_id: 8c3efe458b3b43eaa1399d7740848d3f -> stage3-replay-20260723T065648Z


## Kafka and Spark checkpoint proof

The captured checkpoint evidence shows that replay uses the persisted Spark checkpoint instead of starting a fresh metadata stream from scratch.


In [3]:
print('Kafka replay delta:', manifest['kafka']['delta'])
print('Spark metadata offsets:', manifest['spark']['metadata_offset_before'], '->', manifest['spark']['metadata_offset_after_restart'], '->', manifest['spark']['metadata_offset_after_replay'])
assert manifest['kafka']['delta']['cpg.nodes'] > 0 and manifest['kafka']['delta']['cpg.edges'] >= 0
assert manifest['kafka']['delta']['cpg.metadata'] == 1 and manifest['kafka']['delta']['cpg.errors'] == 0
before, restarted, replayed = (manifest['spark']['metadata_offset_before'], manifest['spark']['metadata_offset_after_restart'], manifest['spark']['metadata_offset_after_replay'])
assert restarted == before and replayed == before + 1


Kafka replay delta: {'cpg.edges': 16, 'cpg.errors': 0, 'cpg.metadata': 1, 'cpg.nodes': 61}
Spark metadata offsets: 138 -> 138 -> 139


## Neo4j cleanup and MongoDB replacement

The final checks verify convergence after replay: stale graph entities are removed in Neo4j, while MongoDB keeps the current metadata document for the replayed `file_id`.


In [4]:
neo4j = manifest['neo4j']
mongo = manifest['mongodb']
print('Neo4j explicit nodes:', neo4j['before']['explicit_nodes'], '->', neo4j['pre_cleanup']['explicit_nodes'], '->', neo4j['after']['explicit_nodes'])
print('Neo4j relationships:', neo4j['before']['edges'], '->', neo4j['pre_cleanup']['edges'], '->', neo4j['after']['edges'])
print('stale deleted:', neo4j['stale_deleted'])
print('MongoDB documents:', mongo['documents_before'], '->', mongo['documents_after_replay'], '; unchanged:', mongo['unchanged_documents'])
assert neo4j['stale_deleted'] == {'nodes': neo4j['pre_cleanup']['stale_nodes'], 'edges': neo4j['pre_cleanup']['stale_edges']}
assert neo4j['after']['duplicate_node_groups'] == neo4j['after']['duplicate_edge_groups'] == 0
assert mongo['documents_after_replay'] == mongo['documents_before']
assert mongo['unchanged_documents'] == mongo['documents_before'] - 1


Neo4j explicit nodes: 133263 -> 133288 -> 133267
Neo4j relationships: 38141 -> 38144 -> 38142
stale deleted: {'edges': 2, 'nodes': 21}
MongoDB documents: 138 -> 138 ; unchanged: 137


## Database UI evidence

```{admonition} Database UI evidence
:class: important

| Store | UI artifact | Meaning |
|---|---|---|
| Neo4j | `neo4j_after_cleanup.png` | Final graph state after stale cleanup. |
| MongoDB | `mongodb_after_replay.png` | Final metadata document after replacement. |
```

Neo4j Browser after stale cleanup:

![Neo4j replay state](../screenshots/replay/neo4j_after_cleanup.png)

Mongo Express after metadata replacement:

![MongoDB replay document](../screenshots/replay/mongodb_after_replay.png)


## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Deterministic file-level replay, Kafka Connect `MERGE`, MongoDB replacement by `file_id`, and the persisted Spark checkpoint form one auditable idempotency chain. |
| Failed | The original workflow used a partial baseline and treated expected Kafka replay events as invalid duplicate IDs. |
| Resolution | The canonical target is now a baseline file, Kafka append-log repetition is measured separately from store duplication, and cleanup runs only after connector lag reaches zero. |
| Limitation | Structural AST IDs may change after edits, so correctness still depends on file-scoped stale cleanup rather than a production-grade semantic diff. |
```
